In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from xgboost import XGBClassifier


In [2]:
app = pd.read_csv(r'C:\Users\DELL1\Downloads\credit card approval prediction dataset\application_record.csv')
credit = pd.read_csv(r'C:\Users\DELL1\Downloads\credit card approval prediction dataset\credit_record.csv')

app.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


In [ ]:
print(app.shape)
app.info()


In [ ]:
credit.head()
print(credit.shape)
credit.info()


In [ ]:
print("Number of people by income type:")
print(app['NAME_INCOME_TYPE'].value_counts())

plt.figure(figsize=(10.0, 6.0))
sns.countplot(x='NAME_INCOME_TYPE', data=app, palette='Set2')


In [ ]:
print("Number of people by education type:")
print(app['NAME_EDUCATION_TYPE'].value_counts())

plt.figure(figsize=(10.0, 6.0))
sns.countplot(x='NAME_EDUCATION_TYPE', data=app, palette='Set2')


In [ ]:
print("Number of people by family status:")
print(app['NAME_FAMILY_STATUS'].value_counts())

plt.figure(figsize=(10.0, 6.0))
sns.countplot(x='NAME_FAMILY_STATUS', data=app, palette='Set2')


In [ ]:
print("Number of people by housing type:")
print(app['NAME_HOUSING_TYPE'].value_counts())

plt.figure(figsize=(10.0, 6.0))
sns.countplot(x='NAME_HOUSING_TYPE', data=app, palette='Set2')


In [ ]:
print("Number of applicants by gender:")
print(app['CODE_GENDER'].value_counts())

plt.figure(figsize=(6.0, 6.0))
sns.countplot(x='CODE_GENDER', data=app, palette='Set2')


In [ ]:
print("Income summary statistics:")
print(app['AMT_INCOME_TOTAL'].describe())

plt.figure(figsize=(10.0, 6.0))
sns.histplot(app['AMT_INCOME_TOTAL'], kde=True, bins=50)


In [11]:
# dropping duplicate rows
app.drop_duplicates(subset=['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
                             'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
                             'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
                             'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
                             'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS'], keep='first', inplace=True)

In [12]:
app.shape

(90085, 18)

In [13]:
app.isnull().sum()

ID                         0
CODE_GENDER                0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
CNT_CHILDREN               0
AMT_INCOME_TOTAL           0
NAME_INCOME_TYPE           0
NAME_EDUCATION_TYPE        0
NAME_FAMILY_STATUS         0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
FLAG_MOBIL                 0
FLAG_WORK_PHONE            0
FLAG_PHONE                 0
FLAG_EMAIL                 0
OCCUPATION_TYPE        27477
CNT_FAM_MEMBERS            0
dtype: int64

In [14]:
print(app.isnull().sum())
print()
print(app.isnull().mean())

ID                         0
CODE_GENDER                0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
CNT_CHILDREN               0
AMT_INCOME_TOTAL           0
NAME_INCOME_TYPE           0
NAME_EDUCATION_TYPE        0
NAME_FAMILY_STATUS         0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
FLAG_MOBIL                 0
FLAG_WORK_PHONE            0
FLAG_PHONE                 0
FLAG_EMAIL                 0
OCCUPATION_TYPE        27477
CNT_FAM_MEMBERS            0
dtype: int64

ID                     0.000000
CODE_GENDER            0.000000
FLAG_OWN_CAR           0.000000
FLAG_OWN_REALTY        0.000000
CNT_CHILDREN           0.000000
AMT_INCOME_TOTAL       0.000000
NAME_INCOME_TYPE       0.000000
NAME_EDUCATION_TYPE    0.000000
NAME_FAMILY_STATUS     0.000000
NAME_HOUSING_TYPE      0.000000
DAYS_BIRTH             0.000000
DAYS_EMPLOYED          0.000000
FLAG_MOBIL             0.000000
FLAG_WORK_PHONE        0.000000
FLAG_PHONE      

In [15]:
app.drop(columns=['OCCUPATION_TYPE'], inplace=True)

In [16]:
app.isnull().sum()

ID                     0
CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_MOBIL             0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
CNT_FAM_MEMBERS        0
dtype: int64

In [17]:
app['CNT_ADULTS'] = app['CNT_FAM_MEMBERS'] - app['CNT_CHILDREN']

In [18]:
app[['CNT_FAM_MEMBERS', 'CNT_CHILDREN', 'CNT_ADULTS']].head()

,CNT_FAM_MEMBERS,CNT_CHILDREN,CNT_ADULTS
0,2.0,0,2.0
2,2.0,0,2.0
3,1.0,0,1.0
7,1.0,0,1.0
10,2.0,0,2.0


In [19]:
app['DAYS_BIRTH'] = abs(app['DAYS_BIRTH'])
app['DAYS_EMPLOYED'] = abs(app['DAYS_EMPLOYED'])

In [20]:
app[['DAYS_BIRTH', 'DAYS_EMPLOYED']].describe()

,DAYS_BIRTH,DAYS_EMPLOYED
count,90085.000000,90085.000000
mean,15901.614164,65614.048743
std,4255.481856,137602.429385
min,7489.000000,12.000000
25%,12339.000000,985.000000
50%,15566.000000,2296.000000
75%,19429.000000,5544.000000
max,25201.000000,365243.000000


In [21]:
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, 0)

In [22]:
app['DAYS_EMPLOYED'].describe()

count    90085.000000
mean      2012.262197
std       2294.627856
min          0.000000
25%        324.000000
50%       1299.000000
75%       2870.000000
max      17531.000000
Name: DAYS_EMPLOYED, dtype: float64

In [23]:
app.drop(columns=['FLAG_MOBIL', 'CNT_CHILDREN'], inplace=True)

In [24]:
app.columns.tolist()

['ID',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'AMT_INCOME_TOTAL',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'CNT_ADULTS']

In [ ]:
cg = LabelEncoder()
oc = LabelEncoder()
own_r = LabelEncoder()
it = LabelEncoder()
et = LabelEncoder()
fs = LabelEncoder()
ht = LabelEncoder()

app['CODE_GENDER'] = cg.fit_transform(app['CODE_GENDER'])
app['FLAG_OWN_CAR'] = oc.fit_transform(app['FLAG_OWN_CAR'])
app['FLAG_OWN_REALTY'] = own_r.fit_transform(app['FLAG_OWN_REALTY'])
app['NAME_INCOME_TYPE'] = it.fit_transform(app['NAME_INCOME_TYPE'])
app['NAME_EDUCATION_TYPE'] = et.fit_transform(app['NAME_EDUCATION_TYPE'])
app['NAME_FAMILY_STATUS'] = fs.fit_transform(app['NAME_FAMILY_STATUS'])
app['NAME_HOUSING_TYPE'] = ht.fit_transform(app['NAME_HOUSING_TYPE'])


In [26]:
app.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,CNT_ADULTS
0,5008804,1,1,1,427500.0,4,1,0,4,12005,4542,1,0,0,2.0,2.0
2,5008806,1,1,1,112500.0,4,4,1,1,21474,1134,0,0,0,2.0,2.0
3,5008808,0,0,1,270000.0,0,4,3,1,19110,3051,0,1,1,1.0,1.0
7,5008812,0,0,1,283500.0,1,1,2,1,22464,0,0,0,0,1.0,1.0
10,5008815,1,1,1,270000.0,4,1,1,1,16872,769,1,1,1,2.0,2.0


In [27]:
credit_grouped = credit.groupby('ID')['MONTHS_BALANCE'].agg(['min', 'max']).reset_index()
credit_grouped.rename(columns={'min': 'open_month', 'max': 'end_month'}, inplace=True)
credit_grouped['window'] = credit_grouped['end_month'] - credit_grouped['open_month']

credit_grouped.head()

,ID,open_month,end_month,window
0,5001711,-3,0,3
1,5001712,-18,0,18
2,5001713,-21,0,21
3,5001714,-14,0,14
4,5001715,-59,0,59


In [28]:
credit_grouped.shape

(45985, 4)

In [29]:
def to_binary(status):
    if status in ['0', 'X', 'C']:   # good payment or no loan record -> Approved (1)
        return 1
    else:                            # overdue, bad debt -> Not Approved (0)
        return 0

credit['STATUS_BIN'] = credit['STATUS'].apply(to_binary)
print(credit['STATUS_BIN'].value_counts())

STATUS_BIN
1    1034381
0      14194
Name: count, dtype: int64


In [30]:
credit_status = credit.groupby('ID')['STATUS_BIN'].min().reset_index()
credit_status.head()

,ID,STATUS_BIN
0,5001711,1
1,5001712,1
2,5001713,1
3,5001714,1
4,5001715,1


In [31]:
final_df = app.merge(credit_grouped, on='ID', how='inner')
final_df = final_df.merge(credit_status, on='ID', how='inner')

print(f"Merged shape: {final_df.shape}")
final_df.head()

Merged shape: (9709, 20)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,CNT_ADULTS,open_month,end_month,window,STATUS_BIN
0,5008804,1,1,1,427500.0,4,1,0,4,12005,4542,1,0,0,2.0,2.0,-15,0,15,0
1,5008806,1,1,1,112500.0,4,4,1,1,21474,1134,0,0,0,2.0,2.0,-29,0,29,1
2,5008808,0,0,1,270000.0,0,4,3,1,19110,3051,0,1,1,1.0,1.0,-4,0,4,1
3,5008812,0,0,1,283500.0,1,1,2,1,22464,0,0,0,0,1.0,1.0,-20,-4,16,1
4,5008815,1,1,1,270000.0,4,1,1,1,16872,769,1,1,1,2.0,2.0,-5,0,5,1


In [ ]:
X = final_df.drop(columns=['STATUS_BIN'])
y = final_df['STATUS_BIN']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


In [ ]:
def logistic_reg(x_train, x_test, y_tr, y_te):
    """
    Builds, trains, tests, and evaluates a Logistic Regression classification model.
    """
    print("Initializing Logistic Regression classifier...")
    model = LogisticRegression(random_state=42, max_iter=1000)

    print("Training the Logistic Regression model on training data...")
    model.fit(x_train, y_tr)

    print("Generating predictions on the unseen test dataset...")
    preds = model.predict(x_test)

    print("\n--- Model Evaluation Results ---")
    cm = confusion_matrix(y_te, preds)
    print("\nConfusion Matrix:")
    print(cm)

    plt.figure(figsize=(10.0, 6.0))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix - Logistic Regression')
    plt.show()

    print("\nClassification Report:")
    print(classification_report(y_te, preds))

    return model, preds

lr_model, y_pred = logistic_reg(X_train, X_test, y_train, y_test)


In [ ]:
def random_forest(x_train, x_test, y_tr, y_te):
    """
    Builds, trains, and tests a Random Forest classification model,
    returning performance metrics.
    """
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    print("\nTraining Random Forest model...")
    model.fit(x_train, y_tr)

    print("Generating predictions...")
    preds = model.predict(x_test)

    print("\n" + "="*40)
    print("Random Forest Model Evaluation")
    print("="*40)

    print("\n--- Classification Report ---")
    print(classification_report(y_te, preds))

    cm = confusion_matrix(y_te, preds)
    print("\nConfusion Matrix:")
    print(cm)

    plt.figure(figsize=(10.0, 6.0))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix - Random Forest')
    plt.show()

    return model, preds

rf_model, y_pred_rf = random_forest(X_train, X_test, y_train, y_test)


In [ ]:
def d_tree(x_train, x_test, y_tr, y_te):
    model = DecisionTreeClassifier(random_state=42)
    model.fit(x_train, y_tr)
    preds = model.predict(x_test)

    print('*** DecisionTreeClassifier ***')
    print('Confusion Matrix')
    print(confusion_matrix(y_te, preds))
    print('Classification Report')
    print(classification_report(y_te, preds))

    return model, preds

dt_model, y_pred_dt = d_tree(X_train, X_test, y_train, y_test)


In [ ]:
def xgboost_model(x_train, x_test, y_tr, y_te):
    model = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
    model.fit(x_train, y_tr)
    preds = model.predict(x_test)

    print('*** XGBoost Classifier ***')
    print('Confusion Matrix')
    print(confusion_matrix(y_te, preds))
    print('Classification Report')
    print(classification_report(y_te, preds))

    return model, preds

xgb_model, y_pred_xgb = xgboost_model(X_train, X_test, y_train, y_test)


In [ ]:
results = {
    'Logistic Regression': y_pred,
    'Random Forest': y_pred_rf,
    'Decision Tree': y_pred_dt,
    'XGBoost': y_pred_xgb
}

comparison = []
for name, pred_values in results.items():
    acc = accuracy_score(y_test, pred_values)
    f1 = f1_score(y_test, pred_values)
    comparison.append({'Model': name, 'Accuracy': acc, 'F1 Score': f1})

comparison_df = pd.DataFrame(comparison).sort_values(by='F1 Score', ascending=False)
print(comparison_df)


In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

print("Model saved successfully as model.pkl")


In [39]:
with open('encoders.pkl', 'wb') as f:
    pickle.dump({'gender': cg, 'own_car': oc, 'own_realty': own_r,
                 'income_type': it, 'education_type': et,
                 'family_status': fs, 'housing_type': ht}, f)

print("Encoders saved successfully as encoders.pkl")

Encoders saved successfully as encoders.pkl


In [40]:
print(X_train.columns.tolist())

['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'CNT_ADULTS', 'open_month', 'end_month', 'window']


In [ ]:
# Drop ID entirely - it's not a real predictive feature
X = final_df.drop(columns=['STATUS_BIN', 'ID'])
y = final_df['STATUS_BIN']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


In [ ]:
pickle.dump(lr_model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))


In [ ]:
with open('encoders.pkl', 'wb') as f:
    pickle.dump({'gender': cg, 'own_car': oc, 'own_realty': own_r,
                 'income_type': it, 'education_type': et,
                 'family_status': fs, 'housing_type': ht}, f)


In [55]:
rejected = final_df[final_df['STATUS_BIN'] == 0]
rejected.describe()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,CNT_ADULTS,open_month,end_month,window,STATUS_BIN
count,1.283000e+03,1283.000000,1283.000000,1283.000000,1.283000e+03,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.000000,1283.0
mean,5.076145e+06,0.367108,0.359314,0.637568,1.860070e+05,2.391270,3.106002,1.398285,1.310210,15516.801247,2013.523772,0.214341,0.276695,0.092751,2.199532,1.753702,-30.575994,-5.073266,25.502728,0.0
std,4.148555e+04,0.482204,0.479987,0.480890,1.170284e+05,1.754331,1.330544,0.988024,0.976533,4223.272787,2190.631595,0.410525,0.447539,0.290197,0.924160,0.452218,15.936676,10.226877,15.209067,0.0
min,5.008804e+06,0.000000,0.000000,0.000000,2.700000e+04,0.000000,0.000000,0.000000,0.000000,7705.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-1.000000,-60.000000,-52.000000,1.000000,0.0
25%,5.036651e+06,0.000000,0.000000,0.000000,1.125000e+05,1.000000,1.000000,1.000000,1.000000,11924.500000,430.500000,0.000000,0.000000,0.000000,2.000000,2.000000,-44.000000,-5.000000,13.000000,0.0
50%,5.069377e+06,0.000000,0.000000,1.000000,1.575000e+05,4.000000,4.000000,1.000000,1.000000,15047.000000,1402.000000,0.000000,0.000000,0.000000,2.000000,2.000000,-30.000000,0.000000,22.000000,0.0
75%,5.113405e+06,1.000000,1.000000,1.000000,2.250000e+05,4.000000,4.000000,2.000000,1.000000,18909.000000,2775.500000,0.000000,1.000000,0.000000,3.000000,2.000000,-16.500000,0.000000,38.000000,0.0
max,5.150459e+06,1.000000,1.000000,1.000000,1.575000e+06,4.000000,4.000000,4.000000,5.000000,25099.000000,14775.000000,1.000000,1.000000,1.000000,9.000000,2.000000,-1.000000,0.000000,60.000000,0.0


In [56]:
rejected.sample(5)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,CNT_ADULTS,open_month,end_month,window,STATUS_BIN
5341,5090098,1,1,1,256500.0,4,1,3,1,9785,1345,0,0,1,1.0,1.0,-37,0,37,0
5950,5091897,0,1,1,67500.0,4,4,1,1,19182,900,0,1,0,2.0,2.0,-7,0,7,0
7516,5115412,0,0,1,495000.0,0,1,3,2,14443,1686,0,1,0,1.0,1.0,-17,0,17,0
1649,5029505,0,0,1,292500.0,1,4,0,1,22720,0,0,1,0,2.0,2.0,-9,0,9,0
7720,5116536,0,0,0,135000.0,0,4,1,2,18524,1707,0,0,0,2.0,2.0,-33,0,33,0


In [ ]:
# Take a real rejected applicant's demographic values, but zero out credit history
# (mimicking what a brand-new applicant looks like in your Flask app)
test_row = final_df.drop(columns=['STATUS_BIN', 'ID']).loc[4027].copy()
test_row['open_month'] = 0
test_row['end_month'] = 0
test_row['window'] = 0

test_array = np.array([test_row.values])
test_scaled = scaler.transform(test_array)
print(lr_model.predict(test_scaled))


In [50]:
# Drop credit-history-derived features - a NEW applicant never has these yet
X = final_df.drop(columns=['STATUS_BIN', 'ID', 'open_month', 'end_month', 'window'])
y = final_df['STATUS_BIN']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.16      0.56      0.25       257
           1       0.89      0.54      0.67      1685

    accuracy                           0.54      1942
   macro avg       0.52      0.55      0.46      1942
weighted avg       0.79      0.54      0.62      1942



In [51]:
pickle.dump(lr_model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

In [52]:
print(pd.Series(y_pred).value_counts())
print(y_test.value_counts())

1    1024
0     918
Name: count, dtype: int64
STATUS_BIN
1    1685
0     257
Name: count, dtype: int64


In [53]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
print(pd.Series(y_pred).value_counts())

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       257
           1       0.87      1.00      0.93      1685

    accuracy                           0.87      1942
   macro avg       0.43      0.50      0.46      1942
weighted avg       0.75      0.87      0.81      1942

1    1942
Name: count, dtype: int64


C:\Users\DELL1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\DELL1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\DELL1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
pickle.dump(lr_model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

In [ ]:
# Logistic Regression with balanced class weights
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

# Random Forest with balanced class weights
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)

print("--- Logistic Regression (balanced) ---")
print(classification_report(y_test, lr_model.predict(X_test_scaled)))

print("--- Random Forest (balanced) ---")
print(classification_report(y_test, rf_model.predict(X_test)))


In [ ]:
pickle.dump(lr_model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
print("Final model saved.")


In [61]:
test_ids = [5341, 5950, 7516, 1649, 7720, 2921, 6697, 4027, 1457, 6413]
X_cols = final_df.drop(columns=['STATUS_BIN', 'ID', 'open_month', 'end_month', 'window']).columns

for idx in test_ids:
    row = final_df.drop(columns=['STATUS_BIN', 'ID', 'open_month', 'end_month', 'window']).loc[idx]
    row_df = pd.DataFrame([row.values], columns=X_cols)
    row_scaled = scaler.transform(row_df)
    pred = lr_model.predict(row_scaled)[0]
    print(f"Row {idx}: predicted = {pred}")

Row 5341: predicted = 0
Row 5950: predicted = 1
Row 7516: predicted = 0
Row 1649: predicted = 1
Row 7720: predicted = 0
Row 2921: predicted = 0
Row 6697: predicted = 0
Row 4027: predicted = 1
Row 1457: predicted = 0
Row 6413: predicted = 1
